In [ ]:
from typing import Annotated, TypedDict, List, Literal
import os
from langgraph.graph import StateGraph, START, END, MessagesState
from langgraph.graph.message import add_messages
from langchain_anthropic import ChatAnthropic
from dotenv import load_dotenv
from IPython.display import Image, display
from langgraph.prebuilt import ToolNode, tools_condition
from langgraph.checkpoint.memory import MemorySaver
from langchain_core.messages import BaseMessage, AIMessage, HumanMessage, SystemMessage
from langchain_core.tools import tool
from langchain.agents import create_agent
from langchain_community.tools.tavily_search import TavilySearchResults
from langchain_core.prompts import ChatPromptTemplate

In [ ]:
class SupervisorState(MessagesState):
    """State for the multi-agent system"""
    next_agent: str = ""
    research_data: str = ""
    analysis: str = ""
    final_report: str = ""
    task_complete: bool = False
    current_task: str = ""

In [ ]:
llm = ChatAnthropic(model='claude-sonnet-4-6')
llm

In [ ]:
def create_supervisor_chain():
    """Create the supervisor decision chain"""
    supervisor_prompt = ChatPromptTemplate.from_messages([
        ("system", """You are a supervisor managing a team of agents:
        
1. Researcher - Gathers information and data
2. Analyst - Analyzes data and provides insights  
3. Writer - Creates reports and summaries

Based on the current state and conversation, decide which agent should work next.
If the task is complete, respond with 'DONE'.

Current state:
- Has research data: {has_research}
- Has analysis: {has_analysis}
- Has report: {has_report}

Respond with ONLY the agent name (researcher/analyst/writer) or 'DONE'.
"""),
        ("human", "{task}")
    ])
    return supervisor_prompt | llm

In [ ]:
def supervisor_agent(state:SupervisorState):
    """Supervisor that manages Researcher, Analyst, and Writer"""
    messages = state["messages"]
    task = messages[-1].content if messages else "No task"
    has_research = bool(state.get("research_data", ""))
    has_analysis = bool(state.get("analysis", ""))
    has_report = bool(state.get("final_report", ""))

    chain = create_supervisor_chain()
    decision = chain.invoke({
        "has_research":has_research,
        "has_analysis":has_analysis,
        "has_report":has_report,
        "task":task,
    })
    decision_text = decision.content.strip().lower()

    if "researcher" in decision_text or not has_research:
        next_agent = "researcher"
        supervisor_msg = "Supervisor: Lets do with research, handing off to Researcher..."
    elif "analyst" in decision_text or (not has_analysis and has_research):
        next_agent = "analyst"
        supervisor_msg = "Supervisor: Lets do with analysis, handing off to Analyst..."
    elif "writer" in decision_text or (not has_report and has_analysis):
        next_agent = "writer"
        supervisor_msg = "Supervisor: Lets do with report, handing off to Writer..."
    else:
        next_agent = "end"        
        supervisor_msg = "Supervisor: All tasks complete! Great work team."

    return {
        "messages": [AIMessage(content = supervisor_msg)],
        "next_agent": next_agent,
        "current_task": task
    }

    

In [ ]:
def researcher_agent(state:SupervisorState):
    """Anthropic Researcher"""
    task = state.get("current_task", "research_topic")
    research_prompt = f"""As a research specialist, provide comprehensive information about: {task}

    Include:
    1. Key facts and background
    2. Current trends or developments
    3. Important statistics or data points
    4. Notable examples or case studies
    
    Be concise but thorough."""

    msg = HumanMessage(content=research_prompt)
    content = llm.invoke(input=msg)
    data = content.content

    return {
        "messages":f"Researcher: I completed the research on {task}\n\nKey Findings:\n{data[:500]}...\n",
        "next_agent": "supervisor",
        "research_data":data
    }

In [ ]:
def analyst_agent(state:SupervisorState):
    """Anthropic Analyst"""
    task = state.get("current_task", "analyze")
    research_data = state.get("research_data", "")
    analysis_prompt = f"""As a data analyst, analyze this research data and provide insights:

    Research Data:
    {research_data}

    Provide:
    1. Key insights and patterns
    2. Strategic implications
    3. Risks and opportunities
    4. Recommendations

    Focus on actionable insights related to: {task}"""

    msg = HumanMessage(content=analysis_prompt)
    content = llm.invoke(input=msg)
    data = content.content

    return {
        "messages":f"Analyst: I completed the analysis on {task}\n\Analysis:\n{data[:500]}...\n",
        "next_agent": "supervisor",
        "analysis":data
    }

In [ ]:
def writer_agent(state:SupervisorState):
    """Anthropic Writer"""
    task = state.get("current_task", "write")
    research_data = state.get("research_data", "")
    analysis = state.get("analysis", "")
    analysis_prompt =     writing_prompt = f"""As a professional writer, create an executive report based on:

Task: {task}

Research Findings:
{research_data[:1000]}

Analysis:
{analysis[:1000]}

Create a well-structured report with:
1. Executive Summary
2. Key Findings  
3. Analysis & Insights
4. Recommendations
5. Conclusion

Keep it professional and concise."""

    msg = HumanMessage(content=analysis_prompt)
    content = llm.invoke(input=msg)
    data = content.content

    return {
        "messages":f"Writer: I completed the report on {task}\n\Report:\n{data[:500]}...\n",
        "next_agent": "supervisor",
        "final_report":data
    }

In [ ]:
def router(state:SupervisorState):
    next_agent = state.get("next_agent", "supervisor")
    if next_agent == "end" or state.get("task_complete", False):
        return END
    elif next_agent in ["supervisor", "researcher", "analyst", "writer"]:
        return next_agent
    return "supervisor"

In [ ]:
workflow = StateGraph(MessagesState)
workflow.add_node("supervisor", supervisor_agent)
workflow.add_node("researcher", researcher_agent)
workflow.add_node("analyst", analyst_agent)
workflow.add_node("writer", writer_agent)

workflow.add_conditional_edges("supervisor", router, {
            "supervisor": "supervisor",
            "researcher": "researcher",
            "analyst": "analyst",
            "writer": "writer",
            END: END
        })
workflow.add_conditional_edges("researcher", router, {
            "supervisor": "supervisor",
            "researcher": "researcher",
            "analyst": "analyst",
            "writer": "writer",
            END: END
        })
workflow.add_conditional_edges("analyst", router, {
            "supervisor": "supervisor",
            "researcher": "researcher",
            "analyst": "analyst",
            "writer": "writer",
            END: END
        })
workflow.add_conditional_edges("writer", router, {
            "supervisor": "supervisor",
            "researcher": "researcher",
            "analyst": "analyst",
            "writer": "writer",
            END: END
        })

In [ ]:
graph=workflow.compile()
graph

In [ ]:
response=graph.invoke(HumanMessage(content="What are the benefits and risks of AI in healthcare?"))
response['final_report']